# Nested decision fits for the cost-aware ask rule

The cost-aware rule estimates regret from gradient-boosting fits whose test folds are 0–2, and the realized regret is
measured on folds 3–4. In the original five-fold design the decision fits were trained on rows that include folds 3–4.
This run refits the decision stage on folds 0–2 only (each fold tested by a model trained on the other two), so no
evaluation row enters any decision fit, and refits fold 0 of the original design to check that this environment
reproduces the original fits. Every fit is saved to Drive as soon as it finishes. After a disconnect, run cells 1 and
3 again and then the cell that was running: finished fits are skipped.

**scikit-learn 1.9.1 is required.** The original fits used it; Colab's default version gives different fits. Cell 2
installs it and cell 3 refuses to continue with any other version.

**Runtime:** CPU (the default) is enough. About 1 min for cell 2, 25–45 min for cell 5 and 15–25 min for cell 6.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/task_inference_nested'   # change if you uploaded elsewhere
RESULTS = BASE + '/results'
import os
for f in ('config.json', 'targets.csv', 'frozen_fold0.csv', 'nested_lib.py', 'data'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f}'
print('ok')

## 2. Install scikit-learn 1.9.1 (~1 min)

If Colab asks you to restart the session after this, click **Restart**, then run cell 1 and continue with cell 3.

In [ ]:
%pip install -q "scikit-learn==1.9.1"

## 3. Load the library and check the version

This cell stops with a message if scikit-learn is not 1.9.1. Then use Runtime › Restart session and run cell 1 and
this cell again.

In [ ]:
import sys, importlib
sys.path.insert(0, BASE)
import nested_lib
importlib.reload(nested_lib)
cfg, targets = nested_lib.load_bundle(BASE)
nested_lib.check_sklearn(cfg)
print(len(targets), 'targets;', cfg)
print(nested_lib.versions())

## 4. Smoke test (under a minute, writes to /content)

I010 is a target whose fits depend on the scikit-learn version. The output must say `15 ok of ...`, `evaluation rows
in any nested training set: 0` and `5/5 identical`. If it does not, stop and send the output.

In [ ]:
nested_lib.run(BASE, '/content/smoke_nested', design='nested', only={'I010'})
nested_lib.run(BASE, '/content/smoke_nested', design='fidelity', only={'I010'})
_ = nested_lib.summarize('/content/smoke_nested', BASE)

## 5. Nested decision fits (~25–45 min, resumable)

In [ ]:
nested_lib.run(BASE, RESULTS, design='nested')

## 6. Fidelity fits: fold 0 of the original design (~15–25 min, resumable)

The largest target (isolet) comes last and takes several minutes on its own.

In [ ]:
nested_lib.run(BASE, RESULTS, design='fidelity')

## 7. Summary and zip

The summary should say `525 ok of 525 expected`, `evaluation rows in any nested training set: 0` and
`175/175 identical`. Download `nested_results.zip` (also saved in the Drive folder) and put it in the project folder.

In [ ]:
import shutil
_ = nested_lib.summarize(RESULTS, BASE)
zip_path = shutil.make_archive(BASE + '/nested_results', 'zip', RESULTS)
print('wrote', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:  # the zip is also in the Drive folder
    print(f'automatic download failed ({e}); download nested_results.zip from the Drive folder')

### Optional: retry fits that failed

Only if cell 7 lists errors that look like a disconnect or out-of-memory. Then run cell 7 again.

In [ ]:
nested_lib.run(BASE, RESULTS, design='nested', retry_errors=True)
nested_lib.run(BASE, RESULTS, design='fidelity', retry_errors=True)